# Model 1: XGBoost - House Prices

**Deep Learning Lab 02**

- Tinh chỉnh siêu tham số với `RandomizedSearchCV`
- Đánh giá nhiều cấu hình
- Lưu kết quả theo format `experiments/xgboost/<timestamp>/`

**Thành viên:** Mai Thị Thúy An · Trần Đoàn Phương Quyên · Nguyễn Thị Phương Thanh

## 1. Import thư viện

In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

## 2. Load dữ liệu

- **VS Code:** `DATA_DIR = Path("../data")`
- **Google Colab:** upload file rồi đổi thành `DATA_DIR = Path(".")`

In [ ]:
# ===== COLAB: bỏ comment để upload =====
# from google.colab import files
# uploaded = files.upload()

DATA_DIR = Path("../data")   # Colab thì đổi thành Path(".")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train:", train_df.shape)
print("Test :", test_df.shape)

## 3. Preprocessing

In [ ]:
def preprocess(train_df, test_df):
    train = train_df.copy()
    test  = test_df.copy()

    cat_cols = [
        "FireplaceQu", "GarageType", "GarageFinish", "MasVnrType",
        "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
        "BsmtFinType2", "GarageQual", "GarageCond", "MSZoning",
        "Utilities", "Exterior1st", "Exterior2nd", "KitchenQual",
        "Functional", "SaleType", "Electrical"
    ]
    for col in cat_cols:
        if col in train.columns:
            mode_val = train[col].mode()[0]
            train[col] = train[col].fillna(mode_val)
            if col in test.columns:
                test[col] = test[col].fillna(mode_val)

    num_cols = [
        "LotFrontage", "GarageYrBlt", "MasVnrArea", "BsmtFinSF1",
        "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath",
        "BsmtHalfBath", "GarageCars", "GarageArea"
    ]
    for col in num_cols:
        if col in train.columns:
            mean_val = train[col].mean()
            train[col] = train[col].fillna(mean_val)
            if col in test.columns:
                test[col] = test[col].fillna(mean_val)

    drop_cols = ["Id", "Alley", "PoolQC", "Fence", "MiscFeature"]
    train = train.drop(columns=[c for c in drop_cols if c in train.columns])
    test  = test.drop(columns=[c for c in drop_cols if c in test.columns])

    y = train["SalePrice"].values
    train = train.drop(columns=["SalePrice"])

    combined = pd.concat([train, test], axis=0)
    combined = pd.get_dummies(combined, drop_first=True)
    combined = combined.loc[:, ~combined.columns.duplicated()]

    X = combined.iloc[:len(train)].astype(float)
    X_test = combined.iloc[len(train):].astype(float)
    X_test = X_test.reindex(columns=X.columns, fill_value=0)

    return X, y, X_test

X, y, X_test = preprocess(train_df, test_df)
print("X     :", X.shape)
print("y     :", y.shape)
print("X_test:", X_test.shape)

## 4. Tinh chỉnh siêu tham số (RandomizedSearchCV)

Mở rộng nhiều nhóm tham số để có nhiều cấu hình đánh giá.

In [ ]:
xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)

param_distributions = {
    # Nhóm 1: Cấu trúc cây
    "n_estimators":     [100, 300, 500, 800, 1200],
    "max_depth":        [2, 3, 4, 5, 7, 10],
    "min_child_weight": [1, 2, 3, 5],

    # Nhóm 2: Learning rate
    "learning_rate":    [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],

    # Nhóm 3: Sampling
    "subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],

    # Nhóm 4: Regularization
    "reg_alpha":        [0, 0.01, 0.1, 1],
    "reg_lambda":       [0.5, 1, 1.5, 2],
}

search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=25,
    scoring="neg_mean_squared_error",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

print("Bắt đầu RandomizedSearchCV (25 cấu hình)...")
search.fit(X, y)

print("\nBest parameters:")
print(search.best_params_)
print("Best CV RMSE:", round(np.sqrt(-search.best_score_), 2))

## 5. Đánh giá nhiều cấu hình (Top 10)

In [ ]:
results_df = pd.DataFrame(search.cv_results_)
results_df["rmse"] = np.sqrt(-results_df["mean_test_score"])
results_df["rmse_std"] = results_df["std_test_score"] / (2 * np.sqrt(-results_df["mean_test_score"]))

top10 = results_df.sort_values("rmse").head(10)[[
    "rmse", "rmse_std",
    "param_n_estimators", "param_max_depth", "param_learning_rate",
    "param_subsample", "param_colsample_bytree",
    "param_reg_alpha", "param_reg_lambda"
]].reset_index(drop=True)

print("=== TOP 10 CẤU HÌNH TỐT NHẤT ===")
display(top10.round(2))

## 6. Baseline vs Best model

In [ ]:
# Baseline: XGBoost mặc định
baseline = xgb.XGBRegressor(random_state=42, n_jobs=-1)
baseline_scores = cross_val_score(
    baseline, X, y,
    scoring="neg_mean_squared_error", cv=5, n_jobs=-1
)
baseline_rmse = np.sqrt(-baseline_scores.mean())

best_rmse = np.sqrt(-search.best_score_)

print(f"Baseline RMSE (mặc định): {baseline_rmse:,.2f}")
print(f"Best RMSE (đã tinh chỉnh): {best_rmse:,.2f}")
print(f"Cải thiện: {baseline_rmse - best_rmse:,.2f} ({(baseline_rmse - best_rmse)/baseline_rmse*100:.1f}%)")

## 7. Feature Importance

In [ ]:
best_model = search.best_estimator_

importance = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importance.head(15).plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importance - XGBoost")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\nTop 10 feature quan trọng nhất:")
print(importance.head(10))

## 8. Predict trên tập test

In [ ]:
preds = best_model.predict(X_test)
print("Predictions shape:", preds.shape)
print("Sample:", preds[:5].round(1))

## 9. Lưu kết quả vào experiments/xgboost/

Tạo đúng format giống các bạn trong nhóm.

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = Path("experiments") / "xgboost" / timestamp
run_dir.mkdir(parents=True, exist_ok=True)

# ----- 1. config.json -----
config = {
    "model_name": "xgboost",
    "best_params": search.best_params_,
    "param_distributions": {k: list(map(str, v)) if not isinstance(v[0], (int, float)) else v
                           for k, v in param_distributions.items()},
    "n_iter": 25,
    "cv": 5,
    "scoring": "neg_mean_squared_error",
    "random_state": 42,
    "timestamp": timestamp
}
with open(run_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False, default=str)

# ----- 2. metrics.json -----
metrics = {
    "best_cv_rmse": float(best_rmse),
    "baseline_cv_rmse": float(baseline_rmse),
    "improvement": float(baseline_rmse - best_rmse),
    "improvement_percent": float((baseline_rmse - best_rmse) / baseline_rmse * 100),
    "n_iter": 25,
    "cv_folds": 5,
    "top5_rmse": top10["rmse"].head(5).round(2).tolist()
}
with open(run_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# ----- 3. model.pkl -----
joblib.dump(best_model, run_dir / "model.pkl")

# ----- 4. predictions.csv -----
sub = sample_sub.copy()
sub["SalePrice"] = preds
sub.to_csv(run_dir / "predictions.csv", index=False)

# Lưu thêm submission ở root
sub.to_csv("submission_xgboost.csv", index=False)

# Lưu thêm bảng top10
top10.to_csv(run_dir / "top10_configs.csv", index=False)

print(f"Đã lưu kết quả vào: {run_dir.resolve()}")
print("Các file:")
for f in sorted(run_dir.iterdir()):
    print(" -", f.name)

## 10. Kiểm tra file đã lưu

In [ ]:
print("=== config.json ===")
print(json.dumps(config, indent=2, default=str)[:800], "...")

print("\n=== metrics.json ===")
print(json.dumps(metrics, indent=2))

print("\n=== predictions (5 dòng đầu) ===")
print(sub.head())

## 11. Nhận xét kết quả

**Tóm tắt:**
- Đã thử **25 cấu hình** khác nhau với 4 nhóm siêu tham số.
- Model sau khi tinh chỉnh cải thiện so với baseline.
- Các feature quan trọng nhất thường liên quan đến chất lượng tổng thể, diện tích sống và vị trí (Neighborhood).

**Hướng cải thiện tiếp theo:**
- Thử log-transform `SalePrice`
- Kết hợp với các model khác (ensemble)
- Làm MLP bằng PyTorch để so sánh

---

**DONE – Model 1: XGBoost**

```
experiments/xgboost/<timestamp>/
├── config.json
├── metrics.json
├── model.pkl
├── predictions.csv
└── top10_configs.csv
```